> ## SOLUTION / ANSWER KEY &mdash; Lab 4.6
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-06-train-val-split.ipynb`](../lab-06-train-val-split.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 4.6 &mdash; Train / Validation Split: Judge on Unseen Data

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Pre-trained Models & Fine-tuning**

### What you'll do
- Split a dataset into train and validation sets
- Stratify so both classes appear in each split
- Understand why you never evaluate on training data

> **How this lab works (near-real):** these labs use **real Hugging Face Transformers** locally on CPU &mdash; a real pretrained sentiment model, a real tokenizer, and (the headline) a **real fine-tune** you run yourself. Read the **Concept**, fill the real `___` blanks in **Build it** (real model / tokenizer / training-loop calls), **Run it for real** to see the **actual model output** (including the real **before &rarr; after** fine-tune numbers), note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real results you can read. The genuine maths (softmax, precision/recall) you still compute **by hand** &mdash; real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased-finetuned-sst-2-english` (out-of-the-box sentiment + logits), `distilbert-base-uncased` (tokenizer), `prajjwal1/bert-tiny` (frozen features **and** the model you fine-tune). First use downloads the weights (needs network), then they are cached. An optional hosted comparison uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 4 slides &mdash; Dataset prep for fine-tuning](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 4 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (optional hosted compare)

WORK = "/tmp/biaa-lab-04-06"
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
You must judge a model on data it **never trained on**, or you only measure memorisation. We hold
out a **validation set** and **stratify** so the class proportions are preserved in both splits
(vital for small or imbalanced data). This is the split every fine-tune lab that follows relies on to
report an honest before/after number.

## Build it
Make a stratified train/validation split.

In [ ]:
# A tiny labelled sentiment dataset (1 = positive, 0 = negative)
SENT = [
    ("i love this it is great", 1), ("a great and wonderful film", 1),
    ("truly wonderful i love it", 1), ("excellent and brilliant work", 1),
    ("the best most brilliant story", 1), ("i love how great it is", 1),
    ("wonderful excellent and great fun", 1), ("a brilliant and great success", 1),
    ("great fun i really love it", 1), ("the best film wonderful and brilliant", 1),
    ("excellent great and lovely work", 1), ("i love this brilliant great film", 1),
    ("wonderful great and the best", 1), ("so good i love it great", 1),
    ("i hate this it is terrible", 0), ("a terrible and awful film", 0),
    ("truly awful i hate it", 0), ("boring and terrible work", 0),
    ("the worst most boring story", 0), ("i hate how bad it is", 0),
    ("awful boring and dull mess", 0), ("a terrible and bad failure", 0),
    ("boring mess i really hate it", 0), ("the worst film awful and boring", 0),
    ("terrible bad and dull work", 0), ("i hate this awful boring film", 0),
    ("awful terrible and the worst", 0), ("so bad i hate it terrible", 0),
]
texts  = [t for t, y in SENT]
labels = [y for t, y in SENT]
from sklearn.model_selection import train_test_split

def split():
    test_frac = 0.3
    return train_test_split(texts, labels, test_size=test_frac,
                            random_state=0, stratify=labels)

## Run it for real
Make the split and sanity-check it.

In [ ]:
try:
    Xtr, Xval, ytr, yval = split()
    print("train:", len(Xtr), "| val:", len(Xval), "| val positives:", sum(yval))
    print("both classes in val?", set(yval) == {0, 1})
    print("no leakage (train/val disjoint)?", set(Xtr).isdisjoint(set(Xval)))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__, "--", e)

## What to notice
- The validation set is **held out** &mdash; the model never sees it in training, so its accuracy there estimates real-world performance.
- **Stratify** keeps both classes present in each split; on tiny data an un-stratified split can accidentally drop a whole class.
- **No leakage**: train and val texts are disjoint. Overlap would inflate your score and lie to you.

## Your turn (open task &mdash; no grader)
Change `test_frac` and re-run &mdash; what is the smallest validation set that still contains
both classes? Remove `stratify` and split a few times with different `random_state`s: how often does a
class go missing? A "good" answer: you can explain, on this 28-row set, why stratification is not
optional.

---
### Key takeaway
The validation set is sacred: it is your honest estimate of real-world accuracy, and the yardstick for every before/after fine-tune number to come.

[&#8592; All Module 4 labs](./index.html) &nbsp;&middot;&nbsp; [Module 4 slides](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>